# 03 · Gold model
Business-ready tables built from Silver:
- **`dim_location`**: one row per pin code (where)
- **`fact_survey_response`**: one row per respondent (what was measured)
- **`agg_order_rate_by_segment`** and **`agg_area_summary`**: pre-computed metrics for analysts and dashboards

Gold is rebuilt in full from Silver on each run (overwrite), so rerunning gives the same tables.

In [1]:
from pathlib import Path
from pyspark.sql import functions as F
from food_pipeline.spark_session import get_spark
from food_pipeline import gold

ROOT = Path.cwd().parent
spark = get_spark()
SILVER = (ROOT / "lakehouse" / "silver" / "survey").as_posix()
GOLD = ROOT / "lakehouse" / "gold"

silver = spark.read.format("delta").load(SILVER)
print("silver rows:", silver.count())

silver rows: 285


## Star schema: `dim_location` and `fact_survey_response`

In [2]:
dim_location = gold.build_dim_location(silver)
fact = gold.build_fact_survey_response(silver, dim_location)

print("dim_location rows:", dim_location.count(), "| distinct pin codes:", dim_location.select("pin_code").distinct().count())
print("fact rows:", fact.count(), "| distinct record_hash:", fact.select("record_hash").distinct().count())
dim_location.orderBy("pin_code").show(5)
fact.select("record_hash", "location_key", "age", "age_band", "occupation", "ordered_again").show(5, truncate=12)

dim_location rows: 77 | distinct pin codes: 77


fact rows: 285 | distinct record_hash: 285


+--------------------+--------+--------+---------+
|        location_key|pin_code|latitude|longitude|
+--------------------+--------+--------+---------+
|-2198920900353037468|  560001| 12.9766|  77.5993|
|-4675630011510961850|  560002| 12.9635|  77.5821|
| 8756714206311087513|  560003| 13.0019|  77.5713|
|-3856231036583030744|  560004| 12.9438|  77.5738|
| 9010180830883783315|  560005|  12.998|  77.6227|
+--------------------+--------+--------+---------+
only showing top 5 rows



+------------+------------+---+--------+------------+-------------+
| record_hash|location_key|age|age_band|  occupation|ordered_again|
+------------+------------+---+--------+------------+-------------+
|0026b7d97...|789993535...| 25|   23-26|    Employee|        false|
|046cd3b5c...|-18104653...| 21|   18-22|     Student|         true|
|050411476...|778868989...| 22|   18-22|    Employee|         true|
|0f79133cd...|662764258...| 25|   23-26|    Employee|         true|
|10204639f...|140147710...| 30|   27-30|Self Empl...|        false|
+------------+------------+---+--------+------------+-------------+
only showing top 5 rows



## Integrity checks
- Every fact row must find its location (**left anti join** = fact rows with no match in the dimension).
- Every age must land in a band.
- `location_key` must be unique in the dimension.

In [3]:
orphans = fact.join(dim_location, on="location_key", how="left_anti").count()
no_band = fact.filter(F.col("age_band").isNull()).count()
dup_keys = dim_location.groupBy("location_key").count().filter("count > 1").count()
print("fact rows without a location:", orphans)
print("rows without an age band:", no_band)
print("duplicate location_key values:", dup_keys)
fact.groupBy("age_band").count().orderBy("age_band").show()

fact rows without a location: 0
rows without an age band: 0
duplicate location_key values: 0


+--------+-----+
|age_band|count|
+--------+-----+
|   18-22|   70|
|   23-26|  152|
|   27-30|   43|
|     31+|   20|
+--------+-----+



## Aggregates
Long format: `segment_type` says which column was grouped, `segment_value` which value.
`is_small_sample` flags rates based on fewer than 20 respondents.

In [4]:
agg_segment = gold.build_agg_order_rate_by_segment(fact)
agg_area = gold.build_agg_area_summary(fact, dim_location)

agg_segment.orderBy("segment_type", "segment_order", F.desc("respondents")).show(40, truncate=False)

+--------------------------+-----------------+-------------+-----------+-----------------+----------+----------------------+---------------+
|segment_type              |segment_value    |segment_order|respondents|ordered_again_yes|order_rate|positive_feedback_rate|is_small_sample|
+--------------------------+-----------------+-------------+-----------+-----------------+----------+----------------------+---------------+
|age_band                  |23-26            |NULL         |152        |119              |0.783     |0.829                 |false          |
|age_band                  |18-22            |NULL         |70         |62               |0.886     |0.871                 |false          |
|age_band                  |27-30            |NULL         |43         |20               |0.465     |0.674                 |false          |
|age_band                  |31+              |NULL         |20         |16               |0.8       |0.75                  |false          |
|customer_typ

In [5]:
print("areas:", agg_area.count(), "| areas with a small sample:", agg_area.filter("is_small_sample").count())
print("respondents across areas:", agg_area.agg(F.sum("respondents")).first()[0])
agg_area.orderBy(F.desc("respondents")).show(10)

areas: 77 | areas with a small sample: 76


respondents across areas: 285


+--------------------+--------+--------+---------+-----------+-----------------+----------+----------------------+---------------+
|        location_key|pin_code|latitude|longitude|respondents|ordered_again_yes|order_rate|positive_feedback_rate|is_small_sample|
+--------------------+--------+--------+---------+-----------+-----------------+----------+----------------------+---------------+
|  789993535777020711|  560009|  12.977|  77.5773|         22|               19|     0.864|                 0.864|          false|
|-1810465323722861746|  560038| 12.9783|  77.6408|         11|                9|     0.818|                 0.909|           true|
| 7288474451714754728|  560010|  12.985|  77.5533|          8|                7|     0.875|                 0.875|           true|
|   48341480296498274|  560066| 12.9698|    77.75|          8|                3|     0.375|                   1.0|           true|
| 7243595444073603035|  560034| 12.9261|  77.6221|          7|                6|   

## Questions the Gold tables answer
Only segments with 20 or more respondents are compared, so the rates mean something.

In [6]:
reliable = agg_segment.filter(~F.col("is_small_sample"))
overall = fact.agg(F.round(F.avg(F.col("ordered_again").cast("int")), 3).alias("order_rate"),
                   F.round(F.avg((F.col("feedback") == "Positive").cast("int")), 3).alias("positive_feedback_rate"))
overall.show()

print("Lowest order rate (segments with 20+ respondents):")
reliable.orderBy("order_rate").select("segment_type", "segment_value", "respondents", "order_rate", "positive_feedback_rate").show(5, truncate=False)
print("Highest order rate:")
reliable.orderBy(F.desc("order_rate")).select("segment_type", "segment_value", "respondents", "order_rate", "positive_feedback_rate").show(5, truncate=False)
print("Feedback vs reordering:")
fact.groupBy("feedback").agg(F.count("*").alias("respondents"),
                             F.round(F.avg(F.col("ordered_again").cast("int")), 3).alias("order_rate")).show()

+----------+----------------------+
|order_rate|positive_feedback_rate|
+----------+----------------------+
|     0.761|                 0.811|
+----------+----------------------+

Lowest order rate (segments with 20+ respondents):


+--------------+--------------+-----------+----------+----------------------+
|segment_type  |segment_value |respondents|order_rate|positive_feedback_rate|
+--------------+--------------+-----------+----------+----------------------+
|age_band      |27-30         |43         |0.465     |0.674                 |
|marital_status|Married       |87         |0.609     |0.724                 |
|occupation    |Self Employed |38         |0.632     |0.711                 |
|monthly_income|25001 to 50000|52         |0.635     |0.731                 |
|occupation    |Employee      |94         |0.638     |0.734                 |
+--------------+--------------+-----------+----------+----------------------+
only showing top 5 rows

Highest order rate:


+--------------------------+-------------+-----------+----------+----------------------+
|segment_type              |segment_value|respondents|order_rate|positive_feedback_rate|
+--------------------------+-------------+-----------+----------+----------------------+
|age_band                  |18-22        |70         |0.886     |0.871                 |
|occupation                |Student      |144        |0.875     |0.882                 |
|monthly_income            |No Income    |131        |0.855     |0.885                 |
|marital_status            |Single       |189        |0.847     |0.862                 |
|educational_qualifications|Post Graduate|125        |0.832     |0.872                 |
+--------------------------+-------------+-----------+----------+----------------------+
only showing top 5 rows

Feedback vs reordering:


+--------+-----------+----------+
|feedback|respondents|order_rate|
+--------+-----------+----------+
|Positive|        231|     0.874|
|Negative|         54|     0.278|
+--------+-----------+----------+



## Insights
Based on 285 de-duplicated respondents (overall order rate 0.761, positive feedback 0.811). This is a survey sample, so these are associations, not causes.

1. **Feedback is the strongest signal.** Respondents with positive feedback reorder at 0.874 (n=231); with negative feedback, 0.278 (n=54).
2. **Students reorder most, earners least.** Students 0.875 (n=144) vs employees 0.638 (n=94) and self-employed 0.632 (n=38). Income shows the same split: no income 0.855 (n=131) vs 25,001–50,000 at 0.635 (n=52) and over 50,000 at 0.638 (n=47). These are largely the same people (students have no income), so the two effects can't be separated with this data.
3. **Ages 27–30 are the weakest segment:** order rate 0.465 (n=43), against 0.886 for ages 18–22 (n=70).
4. **Married respondents reorder less:** 0.609 (n=87) vs 0.847 for single respondents (n=189). Age, marriage and employment overlap here too.
5. **Areas can't be compared yet.** 76 of 77 pin codes have fewer than 20 respondents; only 560009 has more (n=22, order rate 0.864). Area-level rates need more data before anyone acts on them.

## Write the Gold tables

In [7]:
tables = {
    "dim_location": dim_location,
    "fact_survey_response": fact,
    "agg_order_rate_by_segment": agg_segment,
    "agg_area_summary": agg_area,
}
for name, df in tables.items():
    path = (GOLD / name).as_posix()
    gold.write_gold(df, path)
    print(f"{name:28} {spark.read.format('delta').load(path).count():>4} rows")

dim_location                   77 rows


fact_survey_response          285 rows


agg_order_rate_by_segment      26 rows


agg_area_summary               77 rows
